# 31장 · RAG 검색 예외와 인용 검사

검색이 끝나서 결과가 비어 있는 경우와 검색 자체가 실패한 경우에는 다른 처리가 필요합니다. 인용 검사는 검색이 돌려준 문서를 기준으로 합니다.

주차 요금 질문의 빈 검색 결과, 검색 시간 초과, 답변의 인용 오류를 각각 재현합니다. 문제가 생긴 위치를 찾아 자료 보충, 재시도, 인용 수정 중 필요한 조치를 고릅니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 빈 검색 결과와 검색 시간 초과

검색을 마쳤는데 빈 목록이 돌아오면 답변 생성을 건너뛸 수 있습니다. 반면 통신이 끊기면 문서가 있는지조차 확인하지 못했습니다.


### 셀 01 · 빈 문서 목록의 종료 조건

가상 도서관의 주차 질문을 로컬 자료에서 찾습니다.


**실행 준비** · 30장의 문서 세 건과 retrieve를 불러옵니다. 아래 준비 셀을 한 번 실행한 뒤 이어지는 예제에서 이번 장의 변경을 확인하세요.

In [ ]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]


In [1]:
found = retrieve('주차 요금은?')
print(found)

[]


빈 목록이 돌아왔으므로 화면에는 자료에서 확인할 수 없다고 씁니다. 원문에 없는 가격을 모델에게 추측시키지 않습니다.

주차 요금 질문의 []는 검색 함수가 정상적으로 돌아온 값입니다. 시간 초과가 나면 목록 자체가 돌아오지 않아 같은 분기를 탈 수 없습니다. 두 경우를 같은 안내로 표시하면 사용자는 자료를 더 주어야 하는지, 잠시 뒤 다시 시도해야 하는지 알기 어렵습니다.


### 셀 02 · 검색 시간 초과 처리

시간 초과만 일으키는 함수로 장애 경로 한 곳을 재현합니다.

이 셀에서 직접 관찰할 것은 TimeoutError입니다. 검색기가 응답하지 않았으므로 문서가 없었다고 판단할 수 없습니다. 서비스로 제공할 때에는 오류를 기록하고 결과를 result=None으로 남길 수 있습니다.


In [2]:
def slow_search(question):
    raise TimeoutError('검색 시간이 초과되었습니다.')

try:
    slow_search('대출 기간은?')
except TimeoutError as error:
    print(type(error).__name__, str(error))

TimeoutError 검색 시간이 초과되었습니다.


이 셀의 except 블록은 예외 이름과 메시지만 출력합니다. 화면을 연결한다면 검색 실패를 따로 기록하고 재시도를 안내할 수 있습니다.


#### 빈 목록과 시간 초과를 구별하는 이유

<mark>빈 목록은 검색을 끝낸 결과이고 TimeoutError는 결과를 받기 전에 일어난 실패입니다.</mark> 주차 질문에는 자료를 보완할 수 있습니다. 시간 초과에는 문서를 더 넣기 전에 검색 호출이 정상으로 돌아왔는지부터 다시 확인해야 합니다.

선수 개념 · 30장 LangGraph의 검색 노드와 답변 노드

#### 빈 검색 결과와 시간 초과의 처리 위치

```mermaid
flowchart TD
q["검색 호출"] --> r{"정상 반환?"}
r -->|빈 목록| n["근거 없음"]
r -->|문서 있음| y["문서 ID 확인"]
r -->|TimeoutError| e["except에서 오류 확인"]
```

두 셀의 반환 경로를 비교한 그림입니다. slow_search가 예외를 내면 결과 목록을 검사하는 줄까지 도달하지 않습니다.

## 검색된 문서 ID로 인용 검증

검색은 성공해도 답변 끝의 ID가 엉뚱할 수 있습니다. 근거 목록과 인용을 먼저 맞추고, 다음으로 그 문장에 적힌 내용까지 원문과 읽어 봅니다.


### 과제 03 · 반환 문서와 인용 ID 대조

대출 질문에서 가져온 문서 목록에 답안의 인용 library-2가 있는지 확인하세요. 참 또는 거짓으로 출력합니다.


In [3]:
found = retrieve('대출 기간은?')
cited = 'library-2'
# retrieved_ids를 만들고 cited가 포함되는지 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>found = retrieve(&#x27;대출 기간은?&#x27;)
cited = &#x27;library-2&#x27;
retrieved_ids = {doc[&#x27;id&#x27;] for doc in found}
print(cited in retrieved_ids)</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>False가 나옵니다. 검색 결과는 library-1이고 인용은 library-2이므로 서비스에서는 이 응답을 인용 오류로 분류할 수 있습니다. 올바른 ID를 달아도 ‘999일’처럼 원문과 다른 답이 가능해 문장 대조가 남습니다.</p>
</details>


### 인용 ID와 문장 검증

<p>library-1을 붙인 답이 “999일”이라고 해도 ID 검사는 통과합니다. 찾은 문서에서 숫자 14와 단위 ‘일’, 적용 대상 ‘도서 대출’을 확인해야 합니다. 연장이나 연체 기간을 설명하는 문장은 이 문서에 없습니다.</p>


### 이번 검색에서 나온 ID만 인용 대조에 사용합니다

아래 기록은 과제에서 검사한 값입니다. 라이브러리에 존재하는 문서라도 이번 요청에서 반환하지 않은 ID이면 포함 검사 결과가 달라집니다.

1. retrieve · 질문 = 대출 기간은?
2. ID 집합 · retrieved_ids = {'library-1'}
3. 포함 검사 · cited = library-2
4. 출력 · cited in retrieved_ids = False


### 과제 04 · 검색 결과에 대출 문서가 있는데 연장 기간을 묻는다면

‘대출 연장은 며칠인가요?’를 retrieve에 전달하세요. 첫 문서의 ID와 본문을 출력하고, 이 자료만으로 연장 기간을 답할 수 있는지 판단하세요.


In [ ]:
question = '대출 연장은 며칠인가요?'
# 이번 검색에서 받은 첫 문서의 ID와 본문을 표시하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>question = &#x27;대출 연장은 며칠인가요?&#x27;
document = retrieve(question)[0]
print(document[&#x27;id&#x27;], document[&#x27;text&#x27;])</code></pre>
<p>실행 결과</p><pre><code>library-1 도서 대출 기간은 14일입니다.</code></pre>
<p>검색기는 ‘대출’이 겹쳐 library-1을 반환합니다. 14일은 처음 빌리는 기간입니다. 연장 규칙이 쓰여 있지 않으므로 “제공된 안내에서 연장 기간은 확인할 수 없습니다”라고 답할 수 있습니다. 검색 성공과 답변 가능 여부를 따로 기록하세요.</p>
</details>


## 돌아보기

사용자가 ‘자료를 더 주어야 할 때’와 ‘잠시 뒤 다시 물어야 할 때’를 구분하는 안내를 한 문장씩 써 보세요.


## 연장 질문의 근거가 충분한지 검사하는 단계 만들기

대출 기간을 검색하면 14일 안내가 돌아옵니다. “대출을 연장하면 며칠 더 쓸 수 있나요?”에는 어떤 근거가 필요한지 따로 판단합니다.

### 셀 06 · 같은 근거 ID로 서로 다른 질문 읽기

연장 질문을 기존 검색기에 넣고 근거 문장 안에 연장에 관한 정보가 있는지 직접 확인하세요.

In [4]:
extension_question = "대출 연장은 며칠인가요?"
extension_docs = retrieve(extension_question, 1)
print(extension_docs[0]["id"], extension_docs[0]["text"])
print("연장" in extension_docs[0]["text"])

library-1 도서 대출 기간은 14일입니다.
False


library-1은 기본 대출 기간을 설명합니다. 여기서 14일을 뽑아 연장 기간이라고 답하면 질문의 조건을 빠뜨립니다.

### 과제 07 · 자료 없음과 조건 미확인을 다르게 돌려주기

검색 결과가 없으면 no_documents, 연장 문구가 없으면 missing_condition, 근거가 있으면 supported를 반환하는 함수를 작성하세요. 수업용 조건어 하나로 확인하는 규칙입니다. 두 종류의 실패를 서로 다른 질문으로 검산하세요.

In [5]:
# assess_extension(question, docs)에서 빈 목록과 조건 누락을 차례로 검사하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def assess_extension(question, docs):
    if not docs:
        return &quot;no_documents&quot;
    if &quot;연장&quot; in question and not any(&quot;연장&quot; in row[&quot;text&quot;] for row in docs):
        return &quot;missing_condition&quot;
    return &quot;supported&quot;

print(assess_extension(extension_question, extension_docs))
print(assess_extension(&quot;주차 요금은?&quot;, retrieve(&quot;주차 요금은?&quot;, 1)))</code></pre>
<p>실행 결과</p><pre><code>missing_condition
no_documents</code></pre>
<p>첫 값은 자료를 찾았지만 연장 규정을 확인할 수 없다는 뜻입니다. 둘째는 찾은 자료 자체가 없습니다. supported도 이 수업의 한 가지 조건을 통과했다는 의미이므로 일반적인 사실 판정기로 사용할 수는 없습니다.</p>
</details>

<mark>문서 ID가 맞더라도 질문 속 조건의 근거까지 있어야 답할 수 있습니다.</mark> 기본 대출 기간과 연장 가능 기간은 같은 단어를 공유하지만 서로 다른 규정입니다.

평가 질문을 만들 때에는 근거가 비는 사례와 조건이 빠지는 사례를 각각 남기세요.